# Jalankan Chat Interface

Pilih **Runtime > Change runtime type > T4 GPU**, lalu jalankan **Run all**. Sel terakhir menampilkan link Gradio sementara dan tetap berjalan sampai runtime dihentikan. Link tersebut publik bagi siapa pun yang mengetahuinya, jadi jangan masukkan data pribadi atau rahasia.

In [ ]:
import os
import subprocess
from pathlib import Path

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip()
assert gpu, "Pilih Runtime > Change runtime type > T4 GPU"

BRANCH = "dev/apiip"
repo_dir = Path("/content/indonesian-legal-compliance-rag")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "https://github.com/FadhilahAfif/indonesian-legal-compliance-rag.git",
            str(repo_dir),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "checkout", BRANCH], cwd=repo_dir, check=True)
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=repo_dir,
        check=True,
    )
os.chdir(repo_dir)
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
print(f"GPU: {gpu}")
print(f"Commit: {subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "uninstall", "-y", "-q",
        "torchvision", "torchcodec", "torchaudio",
    ],
    check=True,
)
subprocess.run([sys.executable, "scripts/check_environment.py"], check=True)

In [ ]:
import subprocess
import sys
from pathlib import Path

from src.rag import REGULATION_BY_FILE

subprocess.run(
    [
        sys.executable, "-m", "gdown", "--folder",
        "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql",
        "-O", "data/raw",
    ],
    check=True,
)
corpus_dir = Path("data/raw")
missing = [name for name in REGULATION_BY_FILE if not (corpus_dir / name).is_file()]
assert not missing, f"PDF corpus tidak lengkap: {missing}"
print(f"Corpus siap: {len(REGULATION_BY_FILE)} PDF")

In [ ]:
import torch

from app import build_app, build_search

assert torch.cuda.is_available(), "CUDA tidak tersedia"
demo = build_app(build_search(corpus_dir, "cuda"))
demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)